# Linear Classification

**Lab 4 · Linear classification with PyTorch**

This lab follows Lab 3 on linear regression. Work in **Google Colab**. No GPU is needed. This notebook creates its own data and does not need variables from the earlier notebooks.

Work from top to bottom, completing and running each answer before continuing. Try each question in its code cell before opening **Show answer**. To use a supplied answer, copy its code into the corresponding code cell and run it. Opening an answer box does not execute its code.

If a later section depends on code from an earlier one and you are stuck, copy in the supplied answer and continue. The goal is that you can complete the whole lab independently.

We will use the following dataset throughout the lab. Our goal is to use two input features, $x_1$ and $x_2$, to classify the observations into class 0 or class 1.

To match the lecture notation, `inputs` will contain the two observed features, while `X` will denote the **design matrix** obtained by adding a first column of ones. This allows the intercept and the two feature weights to be fitted together.


### Reading the data plot

The supplied scatter plot uses the two raw input features. `inputs[:, 0]` selects the first feature and `inputs[:, 1]` the second. The one-dimensional list of observed class labels is `y[:, 0]`, because the stored label tensor has shape `(N, 1)`.

In `plt.scatter(x=inputs[:, 0], y=inputs[:, 1], c=y[:, 0])`, the keyword `y=` means **vertical coordinates**. The separate tensor named `y` contains the observed class labels. `c=` uses those labels to colour the dots. The class label is therefore shown by **colour, not height**.

The matrix `X` used for fitting has one extra first column containing ones. Its rows are therefore

$$
(1,\mathbf{x}_i),
$$

exactly as in the lecture. The added `1` is not another observed feature. It is there so that the same matrix product also includes the intercept.

Run the supplied plotting code. You do not need to reconstruct it.


In [ ]:
import torch as t
import matplotlib.pyplot as plt

t.manual_seed(1)  # Repeat the random draws when rerunning this cell in the same setup.

N = 100
# Generate 50 observations for each class, with two input features per observation.
x_class_0 = 0.5 * t.randn(N // 2, 2) - 1
x_class_1 = t.randn(N // 2, 2) + 1
inputs = t.cat([x_class_0, x_class_1], 0)  # Raw input features; shape (N, 2).

# Labels are in the same order as the observations: 50 zeros, then 50 ones.
y = t.cat([t.zeros(N // 2, 1), t.ones(N // 2, 1)], 0)  # Shape (N, 1).

# Add the constant feature 1 so that the intercept is part of the parameter vector.
X = t.cat([t.ones(N, 1), inputs], dim=1)  # Design matrix; shape (N, 3).

# Horizontal and vertical positions are the two raw input features.
# Colours encode the observed labels.
plt.scatter(x=inputs[:, 0], y=inputs[:, 1], c=y[:, 0])
plt.xlabel('$x_1$')
plt.ylabel('$x_2$')
plt.show()


## 1) Numerical stability in computing the loss

For observation $i$, we use the affine score from the lecture,

$$
f_{\boldsymbol{\theta}}(\mathbf{x}_i)
=
b+\mathbf{x}_i\mathbf{w},
\qquad
\boldsymbol{\theta}
=
\begin{pmatrix}
b\\
\mathbf{w}
\end{pmatrix}.
$$

Here there are two input features, so

$$
\boldsymbol{\theta}
=
\begin{pmatrix}
b\\
w_1\\
w_2
\end{pmatrix}.
$$

Because the first column of the design matrix `X` contains ones, all scores are computed at once as

$$
\mathbf{s}=\mathbf{X}\boldsymbol{\theta}.
$$

The **sigmoid** function $\sigma(s)=1/(1+e^{-s})$ converts each score into the predicted probability of class 1. The probability of class 0 is $1-\sigma(s)=\sigma(-s)$.

In the lecture, the log-loss for a single observation takes a predicted probability $p$ and an observed label $y$:

$$
\ell_{\mathrm{log}}(p,y)
=
-y\log p-(1-y)\log(1-p).
$$

For observation $i$, we use $p_i=\sigma(s_i)$ and the label $y_i$. We therefore evaluate this log-loss for each observation and add the contributions to obtain the empirical logistic loss

$$
\mathcal{L}_{\mathrm{log}}(\boldsymbol{\theta})
=
\sum_{i=1}^N
\ell_{\mathrm{log}}\!\left(\sigma(s_i),y_i\right)
=
-\sum_{i=1}^N
\left[
y_i\log\sigma(s_i)
+
(1-y_i)\log\sigma(-s_i)
\right].
$$

Thus a label $y_i=1$ selects the first term and a label $y_i=0$ selects the second. We use a **sum**, not a mean, as in the lecture.

The Python loss functions we write below will take `theta` as their argument. They will compute the scores from `X @ theta` and use the observed labels stored in `y` to evaluate this sum. In this notebook, the functions will use the data tensors `X` and `y` defined in the setup cell, so we do not pass them as separate arguments. The returned value will therefore be the complete objective $\mathcal{L}_{\mathrm{log}}(\boldsymbol{\theta})$ for the chosen parameters.

### 1.1) Compute the loss using the sigmoid

Write a function `loss_sigmoid(theta)` that takes the full parameter vector and returns the loss. Implement the formula directly using `t.sigmoid` and `t.log`.

**Shapes:** `theta` has shape `(3, 1)`, because it contains the intercept followed by two feature weights. Therefore `X @ theta` and `y` both have shape `(N, 1)`. Keep these column shapes throughout this lab. Mixing `(N,)` and `(N, 1)` would broadcast to `(N, N)` rather than pair each prediction with its own label.


In [ ]:
# Your code here.


<details>
<summary>Show answer — 1.1</summary>

```python
def loss_sigmoid(theta):
    scores = X @ theta  # One score per observation; shape (N, 1).
    # Sigmoid and log act element by element. .sum() adds all N contributions.
    return -(y * t.log(t.sigmoid(scores))
             + (1 - y) * t.log(t.sigmoid(-scores))).sum()
```

</details>


### 1.2) Numerical instability in the naive loss

Test your function with

$$
\boldsymbol{\theta}
=
\begin{pmatrix}
20\\
20\\
20
\end{pmatrix}.
$$

With the default floating-point precision used in this notebook, you should obtain `nan`, meaning **not a number**, rather than a finite loss. This is the numerical problem we want to investigate. It is not a Python exception.

If you obtain a finite value, check whether you are using a different precision before concluding that your implementation is incorrect.


In [ ]:
# Your code here.


<details>
<summary>Show answer — 1.2</summary>

```python
theta = 20 * t.ones(3, 1)
print(loss_sigmoid(theta))
```

</details>


### 1.3) Plotting the naive log-sigmoid calculation

Plot $\log\sigma(s)$ for $s$ between $-100$ and $100$, computing it as `t.log(t.sigmoid(s))`.

Also inspect the computed values near $-100$. The visible part of the curve can look reasonable even when some values are `-inf` (negative infinity): those values cannot be displayed as ordinary points on the graph.

**How to draw it:** once you have the equally spaced scores `s` and their computed values `log_p`, use `plt.plot(s, log_p)`, `plt.xlabel("score s")`, `plt.ylabel("log(sigmoid(s))")`, and `plt.show()`. Both tensors are one-dimensional and have the same length. No NumPy conversion is needed here. The values printed below the graph are important: missing infinite values do not become finite just because the visible line looks reasonable.


In [ ]:
# Your code here.


<details>
<summary>Show answer — 1.3</summary>

```python
s = t.linspace(-100, 100, 100)
log_p = t.log(t.sigmoid(s))

plt.plot(s, log_p)
plt.xlabel('score s')
plt.ylabel('log(sigmoid(s))')
plt.show()

# Look at the calculated values, not only the visible part of the curve.
print(log_p[:5])
```

</details>


### 1.4) Diagnose the problem

Work out why computing $\log\sigma(s)$ in the two separate steps `t.sigmoid(s)` and then `t.log(...)` gives numerical problems for scores $s$ near $-100$.

The key question is: **what floating-point value is actually being passed into the logarithm?**


In [ ]:
# Your code here.


<details>
<summary>Show answer — 1.4</summary>

```python
# In this example, the computed sigmoid becomes exactly zero at -100.
print(t.sigmoid(t.tensor(-50.)))
print(t.sigmoid(t.tensor(-100.)))

# Taking the log of that computed zero gives -inf.
print(t.log(t.tensor(0.)))

# A zero label coefficient times -inf gives nan in floating-point arithmetic.
print(t.tensor(0.) * t.log(t.tensor(0.)))
```

Mathematically, $\sigma(-100)$ is positive and its logarithm is finite. The numerical calculation has lost that small positive value **before** taking the logarithm.

This also explains the `nan` in Question 1.2: some computed log-probabilities become `-inf`, and the loss multiplies them by zero label coefficients (`y` or `1-y`). Numerically, `0 * (-inf)` gives `nan`; adding the contributions does not remove it.

</details>


### 1.5) Use the numerically stable `logsigmoid` function

PyTorch provides [`t.nn.functional.logsigmoid`](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.logsigmoid.html). It computes the same mathematical quantity $\log\sigma(s)$, but avoids losing the result through the intermediate sigmoid value.

Check this function at $-100$ and $-1000$. Then implement a new loss function, named `loss(theta)`, using `logsigmoid`, and check that it returns a finite value for

$$
\boldsymbol{\theta}
=
\begin{pmatrix}
20\\
20\\
20
\end{pmatrix}.
$$

Pass the **scores** to `logsigmoid`, not their sigmoid values. Use this new `loss` function for the rest of the notebook.

We have **not** changed the mathematical loss. We have only changed how it is evaluated numerically.


In [ ]:
# Your code here.


<details>
<summary>Show answer — 1.5</summary>

```python
print(t.nn.functional.logsigmoid(t.tensor(-100.)))
print(t.nn.functional.logsigmoid(t.tensor(-1000.)))

def loss(theta):
    scores = X @ theta
    # Same logistic loss as before, now computing each log-probability directly.
    return -(y * t.nn.functional.logsigmoid(scores)
             + (1 - y) * t.nn.functional.logsigmoid(-scores)).sum()

print(loss(20 * t.ones(3, 1)))
```

</details>


## 2) Implement the gradient, train the classifier, and inspect the result

In the lecture, the gradient of the summed empirical logistic loss is

$$
\nabla\mathcal{L}_{\mathrm{log}}(\boldsymbol{\theta})
=
\mathbf{X}^T
\left(
\mathbf{p}(\boldsymbol{\theta})-\mathbf{y}
\right),
\qquad
\mathbf{p}(\boldsymbol{\theta})
=
\boldsymbol{\sigma}(\mathbf{X}\boldsymbol{\theta}).
$$

Here, $\boldsymbol{\sigma}$ applies the scalar sigmoid separately to each entry of the score column. In the code, `t.sigmoid` performs this entry-by-entry calculation.

This has the same general structure as the regression gradient: the transposed design matrix multiplies a **prediction minus observed value** difference. Here the predictions are probabilities, so the entries are **predicted probability minus observed label**.

### 2.1) Implement the gradient

Write a function `grad(theta)` that implements this formula. Use the stable `loss(theta)` from Question 1.5 for the rest of the lab, but note that the gradient itself uses the sigmoid probabilities.

The result should have the same shape as `theta`, namely `(3, 1)`. The first entry is the derivative with respect to the intercept. The other two entries are the derivatives with respect to the two feature weights.

**Self-check:** `grad(t.zeros(3, 1))` should have shape `(3, 1)` and should be close to

$$
\begin{pmatrix}
0\\
-53.81\\
-51.29
\end{pmatrix}.
$$

The first entry is zero because this dataset contains the same number of observations from each class, so at zero scores the positive and negative label differences cancel in the intercept coordinate.

### 2.2) Train the classifier and inspect what it predicts

Starting from zero parameters, use gradient descent with learning rate `0.01` and 100 updates.

During training, print the stable loss after updates 10, 20, ..., 100.

After training:

1. compute the probability of class 1 for each observation;
2. apply the **plug-in rule**: predict class 1 when the probability is at least `0.5`;
3. count the number of training mistakes;
4. compare that number with the upper bound `loss(theta) / log(2)`.

If your `grad` function still does not work, use the supplied answer below so that you can complete the rest of the lab.


In [ ]:
# Your code here.


<details>
<summary>Show answer — 2</summary>

```python
def grad(theta):
    scores = X @ theta                  # Shape (N, 1).
    probabilities = t.sigmoid(scores)   # Shape (N, 1).
    # X.mT has shape (3, N). The bracket has shape (N, 1).
    return X.mT @ (probabilities - y)

print("Gradient shape:", grad(t.zeros(3, 1)).shape)
print("Gradient at zero:\n", grad(t.zeros(3, 1)))

learning_rate = 0.01
theta = t.zeros(3, 1)
for step in range(1, 101):
    theta = theta - learning_rate * grad(theta)
    # % gives the remainder: this condition holds for step = 10, 20, ..., 100.
    # The update comes first, so step is the number of completed updates.
    if step % 10 == 0:
        print(f"After {step} updates: loss = {loss(theta).item():.6f}")

probabilities = t.sigmoid(X @ theta)
predicted_labels = (probabilities >= 0.5).to(y.dtype)
mistakes = (predicted_labels != y).sum().item()
total_loss = loss(theta).item()
log_two = t.log(t.tensor(2.0)).item()

print("Learned parameters:\n", theta)
# Three columns: probability of class 1, predicted label, observed label.
print(t.cat([probabilities, predicted_labels, y], dim=1)[::10])
print("Number of training mistakes:", mistakes)
print("Summed logistic loss:", total_loss)
print("Upper bound on mistakes:", total_loss / log_two)
```

With the supplied dataset, the learned parameters should be close to

```python
tensor([[1.1950],
        [2.4771],
        [2.6349]])
```

The first entry is the fitted intercept, followed by the two feature weights.

The final summed logistic loss should be about `6.88`. The number of training mistakes should be `3`, while the quantity `loss(theta) / log(2)` should be about `9.92`.

This illustrates two distinct points:

- the logistic loss is **not** the same as the number of mistakes;
- the bound from the lecture is an **upper bound**, so it can be larger than the true number of mistakes.

</details>

### Reading the decision-boundary plot

Run the entire helper cell below after training. It already includes the call that draws your fitted classifier. You do not need to write the grid or contour code. The helper is non-examinable.

As in the initial scatter plot, the dot positions are the two raw input features and their colours are the **observed labels**. The curve is a grid approximation of where the fitted score is zero. The classifier predicts class 1 for a nonnegative score, including a tie, and class 0 for a negative score.

The fitted score is

$$
f_{\boldsymbol{\theta}}(\mathbf{x})
=
b+\mathbf{x}\mathbf{w}.
$$

The plotting helper therefore receives a short function that applies the fitted intercept and feature weights to new raw input points. Re-running only the plotting cell redraws the current classifier. It does **not** perform more training.

The helper uses `t.no_grad()` while calculating the grid predictions because a display does not need derivatives. All tensors here are on CPU. No NumPy conversion is required.


In [ ]:
def plot_decision_boundary(score_function, inputs, Y):
    # Supplied, non-examinable plotting code. All models/data in this lab use CPU.
    x0_grid, x1_grid = t.meshgrid(t.linspace(-4, 4, 300),
                                  t.linspace(-4, 4, 300),
                                  indexing="ij")
    input_grid = t.stack([x0_grid, x1_grid], dim=-1)

    # Display only: do not record a differentiation graph for the grid predictions.
    with t.no_grad():
        c_grid = (score_function(input_grid) >= 0)[..., 0]

    # One contour separates grid labels 0 and 1 and approximates the score-zero boundary.
    if c_grid.any().item() and not c_grid.all().item():
        plt.contour(x0_grid, x1_grid, c_grid, levels=[0.5], linewidths=0.7)
    else:
        print("The model predicts one class everywhere on this grid; no boundary is visible.")

    plt.scatter(inputs[:, 0], inputs[:, 1], c=Y[:, 0])
    plt.xlabel('$x_1$')
    plt.ylabel('$x_2$')
    plt.show()

# theta[0] is the fitted intercept and theta[1:] contains the two feature weights.
plot_decision_boundary(
    lambda new_inputs: theta[0] + new_inputs @ theta[1:],
    inputs,
    y
)
